# Iraq Farms Health — WorldCereal crop types for all of Iraq

Maps **wheat and barley for Iraq's farmland** for one winter season with **ESA WorldCereal's own open-source model** (the same default model as the Arab Dulaim pilot), on the Copernicus Data Space cloud.

**How it works**
1. **Plan** (step 4, free, ~1 min): splits Iraq into ≤ 2,500 km² tiles, keeps only tiles with farmland (FTW field-confidence map), shrinks each to its farmland, takes each tile's winter season from WorldCereal's own crop calendar, and **estimates the credits** — before anything is submitted.
2. **Run a batch** (step 5): submits the next set of tiles that fits your **credit budget** (default 9,000 — your account gets 10,000 free per month), waits, downloads, and shrinks each result to the 4 bands the app needs.
3. **Repeat step 5** (next month, or whenever you have credits) until the status says everything is done.

**Long runs and disconnects:** a batch can take many hours. Everything (plan, job tracking, results) lives in your **Google Drive**, and the cloud jobs keep running if Colab disconnects. Just reconnect, run steps 1–3, then step 5 again — it **resumes the same batch**, it never resubmits or re-bills finished tiles.

Code: [WorldCereal/worldcereal-classification](https://github.com/WorldCereal/worldcereal-classification) pinned at `b739085` (MIT). Farmland mask: [Fields of The World](https://source.coop/ftw) confidence map.

## 1 · Settings
Defaults match the pilot. `SEASON_YEAR = 2026` means the winter season that **ended** in 2026 (Dec 2025 – May 2026 in most of Iraq; WorldCereal's calendar sets each tile). Google will ask for Drive access — **tick every box**.

In [ ]:
# ---- Settings -------------------------------------------------------------
RUN_NAME        = "iraq_winter2026"   # folder in Drive: iraq-farms-health/worldcereal/<RUN_NAME>
SEASON_YEAR     = 2026                # winter season ending in this year (must be over)
TILE_KM         = 50                  # tile size (50 km = 2,500 km², WorldCereal's per-run limit)
FARMLAND_CONF   = 0.22                # FTW confidence counted as farmland (the app's old density threshold)
MIN_FARMLAND_KM2 = 20                 # skip tiles with less farmland than this
CREDIT_BUDGET   = 9000                # max estimated credits per batch (free allowance: 10,000 / month)
PARALLEL_JOBS   = 2                   # jobs running at once on Copernicus
DELETE_RAW_AFTER_COMPACT = True       # keep only the small 4-band copies (saves ~85% Drive space)
SAVE_TO_DRIVE   = True                # strongly recommended: needed to resume across sessions
# ---------------------------------------------------------------------------

import json, os, shlex, subprocess, shutil
def sh(cmd):
    # Run a shell command, streaming its output live (unbuffered) into the notebook.
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    out = []
    for line in p.stdout:
        print(line, end="", flush=True); out.append(line)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {p.returncode}) — see the messages above.")
    return "".join(out)

OUT = f"/content/worldcereal_out/{RUN_NAME}"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = f"/content/drive/MyDrive/iraq-farms-health/worldcereal/{RUN_NAME}"
    except Exception as e:
        SAVE_TO_DRIVE = False
        print(f"⚠️  Google Drive didn't mount ({e}). Re-run this cell and tick EVERY box in Google's pop-up.")
        print("   Without Drive, a disconnect loses the job tracking — don't start a long batch like this.")
os.makedirs(OUT, exist_ok=True)
CFG = os.path.join(OUT, "run_config.json")
json.dump({"out": OUT, "ftw_cog": "https://data.source.coop/ftw/global-data/predictions/confidence/confidence/prue_v1_confidence_global.tif", "tile_km": TILE_KM, "farmland_conf": FARMLAND_CONF,
           "min_farmland_km2": MIN_FARMLAND_KM2, "buffer_m": 1000, "season_year": SEASON_YEAR}, open(CFG, "w"), indent=1)
try:
    tot, used, free = shutil.disk_usage(OUT)
    print(f"Free space where results go: {free/1e9:.1f} GB (a 9,000-credit batch needs ~6 GB while running)")
except Exception:
    pass
print("Run folder:", OUT)

## 2 · Install WorldCereal (≈3–8 min)

In [ ]:
%%bash
set -e
pip -q install uv
if [ ! -d /content/worldcereal-classification ]; then
  git clone -q --filter=blob:none https://github.com/WorldCereal/worldcereal-classification.git /content/worldcereal-classification
fi
cd /content/worldcereal-classification && git checkout -q b739085 && git log -1 --format='WorldCereal code: %h  %ad  %s' --date=short
if [ ! -x /content/wcenv/bin/python ]; then uv venv -q -p 3.12 /content/wcenv; fi
# --python targets the environment explicitly: Colab's uv settings would otherwise send the
# install to Colab's own Python (VIRTUAL_ENV alone is ignored there).
uv pip install -q --python /content/wcenv/bin/python /content/worldcereal-classification rasterio matplotlib
/content/wcenv/bin/python -c "import worldcereal, openeo; print('Installed OK — openeo', openeo.__version__)"

## 3 · Log in to Copernicus
Uses `CDSE_CLIENT_ID` / `CDSE_CLIENT_SECRET` from 🔑 Secrets if present (as in the pilot); otherwise prints a login link.

In [ ]:
import textwrap
try:
    from google.colab import userdata
    cid, sec = userdata.get("CDSE_CLIENT_ID"), userdata.get("CDSE_CLIENT_SECRET")
    os.environ.update({"OPENEO_AUTH_METHOD": "client_credentials",
                       "OPENEO_AUTH_CLIENT_ID_CDSE": cid, "OPENEO_AUTH_CLIENT_SECRET_CDSE": sec,
                       "OPENEO_AUTH_PROVIDER_ID_CDSE": "CDSE"})
    print("Using client credentials from Colab Secrets.")
except Exception:
    os.environ.pop("OPENEO_AUTH_METHOD", None)
    print("No client credentials in Secrets — using the login link.")
login = textwrap.dedent('''
    import os, openeo
    c = openeo.connect("openeo.dataspace.copernicus.eu")
    if os.environ.get("OPENEO_AUTH_METHOD") == "client_credentials":
        c.authenticate_oidc_client_credentials(
            client_id=os.environ["OPENEO_AUTH_CLIENT_ID_CDSE"],
            client_secret=os.environ["OPENEO_AUTH_CLIENT_SECRET_CDSE"],
            provider_id=os.environ.get("OPENEO_AUTH_PROVIDER_ID_CDSE"))
    else:
        c.authenticate_oidc_device(max_poll_time=900, store_refresh_token=True)
    me = c.describe_account()
    print("Logged in as:", me.get("name") or me.get("user_id"))
''')
open("/content/cdse_login.py", "w").write(login)
sh("/content/wcenv/bin/python -u /content/cdse_login.py")

## 4 · Plan the tiles (free — nothing is submitted)
Builds the tile list once and saves it in Drive (re-running reuses it; set `REPLAN = True` to rebuild). **Check the numbers and the map before step 5**, and send them to me.

In [ ]:
%%writefile /content/plan_tiles.py
"""Plan WorldCereal tiles for Iraq's farmland.

python plan_tiles.py <config.json>

1. 50 km (configurable) squares on the UTM grid over Iraq (zones 37/38/39N).
2. Farmland = FTW field-confidence (500 m COG) >= threshold, inside Iraq's border.
   Each square is shrunk to the bounding box of its farmland (+ buffer) and dropped
   if it has too little — so desert costs nothing.
3. Winter season per tile from WorldCereal's own crop calendar (season tc-s1 only;
   the 12-month processing window ends with it, so nothing is in the future).
4. Credit estimate per tile from the pilot (53 credits for 250.6 km²) scaled along
   the hub's published size curve.
Writes plan/grid_all.geojson, plan/plan.json and plan/plan_map.png.
"""
import json
import math
import os
import sys
from datetime import date

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds
from shapely.geometry import box, mapping
from shapely.ops import transform as shp_transform
import pyproj
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

cfg = json.load(open(sys.argv[1]))
OUT = cfg["out"]
PLAN = os.path.join(OUT, "plan")
os.makedirs(PLAN, exist_ok=True)
TILE_M = cfg["tile_km"] * 1000
CONF = cfg["farmland_conf"]
MIN_FARM_KM2 = cfg["min_farmland_km2"]
BUF_M = cfg.get("buffer_m", 1000)
YEAR = cfg["season_year"]

# ---- Iraq border (ships with the WorldCereal package) -----------------------
import worldcereal
wab = os.path.join(os.path.dirname(worldcereal.__file__), "data", "world-administrative-boundaries",
                   "world-administrative-boundaries.geoparquet")
iraq = gpd.read_parquet(wab)
iraq = iraq[iraq["iso3"] == "IRQ"].to_crs(4326)
iraq_geom = iraq.geometry.union_all() if hasattr(iraq.geometry, "union_all") else iraq.geometry.unary_union
W, S, E, N = iraq_geom.bounds
print(f"Iraq border: {W:.2f},{S:.2f} → {E:.2f},{N:.2f}")

# ---- Farmland mask from the FTW confidence COG ------------------------------
src_path = cfg["ftw_cog"]
if src_path.startswith("http"):
    os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
    os.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif")
    src_path = "/vsicurl/" + src_path
with rasterio.open(src_path) as ds:
    rc = ds.crs
    to_r = pyproj.Transformer.from_crs(4326, rc, always_xy=True).transform
    bw, bs, be, bn = shp_transform(to_r, box(W, S, E, N)).bounds
    win = from_bounds(bw, bs, be, bn, ds.transform).round_offsets().round_lengths()
    conf = ds.read(1, window=win, masked=True)
    rt = ds.window_transform(win)
    nod = ds.nodata
print(f"FTW confidence window: {conf.shape[1]}x{conf.shape[0]} px; value range "
      f"{float(conf.min()):.3f}–{float(conf.max()):.3f}; farmland threshold ≥ {CONF}")
inside_iraq = ~geometry_mask([mapping(shp_transform(to_r, iraq_geom))], conf.shape, rt, invert=False)
data = np.ma.filled(conf.astype("float32"), -1.0)
if nod is not None:
    data[data == nod] = -1.0
farm = (data >= CONF) & inside_iraq
rows, cols = np.nonzero(farm)
xs, ys = rasterio.transform.xy(rt, rows, cols)
farm_lon, farm_lat = pyproj.Transformer.from_crs(rc, 4326, always_xy=True).transform(np.asarray(xs), np.asarray(ys))
# pixel area (km²) — measured in an equal-area projection at each pixel
px_w, px_h = abs(rt.a), abs(rt.e)
if rc.is_geographic:
    px_km2 = (px_w * 111.32 * np.cos(np.radians(farm_lat))) * (px_h * 110.57)
else:
    px_km2 = np.full(len(rows), px_w * px_h / 1e6)
print(f"Farmland pixels in Iraq: {len(rows):,}  ≈ {px_km2.sum():,.0f} km² (FTW confidence ≥ {CONF})")

# ---- Squares on each UTM zone, shrunk to farmland ---------------------------
def est_credits(a_km2):
    # Hub curve (250→40, 750→90, 2500→250), extrapolated linearly, scaled by the
    # pilot's real cost (53 vs the hub's 40 for 250 km²).
    pts = [(0, 15), (250, 40), (750, 90), (2500, 250)]
    for (a0, c0), (a1, c1) in zip(pts, pts[1:]):
        if a_km2 <= a1:
            return 1.325 * (c0 + (c1 - c0) * (a_km2 - a0) / (a1 - a0))
    return 1.325 * (250 + (a_km2 - 2500) * (160 / 1750))

tiles = []
for zone in (37, 38, 39):
    epsg = 32600 + zone
    lon0, lon1 = -180 + (zone - 1) * 6, -180 + zone * 6
    sel = (farm_lon >= lon0) & (farm_lon < lon1)
    if not sel.any():
        continue
    fwd = pyproj.Transformer.from_crs(4326, epsg, always_xy=True)
    inv = pyproj.Transformer.from_crs(epsg, 4326, always_xy=True)
    fx, fy = fwd.transform(farm_lon[sel], farm_lat[sel])
    fk = px_km2[sel]
    gx = np.floor(fx / TILE_M).astype(int)
    gy = np.floor(fy / TILE_M).astype(int)
    df = pd.DataFrame({"gx": gx, "gy": gy, "x": fx, "y": fy, "km2": fk})
    for (cx, cy), grp in df.groupby(["gx", "gy"]):
        fkm2 = grp["km2"].sum()
        if fkm2 < MIN_FARM_KM2:
            continue
        x0, y0 = cx * TILE_M, cy * TILE_M
        # farmland bbox inside the square, buffered, clipped to the square
        half = max(px_w, px_h) * (111320 if rc.is_geographic else 1) / 2
        bx0 = max(x0, grp["x"].min() - half - BUF_M); bx1 = min(x0 + TILE_M, grp["x"].max() + half + BUF_M)
        by0 = max(y0, grp["y"].min() - half - BUF_M); by1 = min(y0 + TILE_M, grp["y"].max() + half + BUF_M)
        utm_box = box(bx0, by0, bx1, by1)
        area = utm_box.area / 1e6
        geom = shp_transform(inv.transform, utm_box)
        tiles.append({"tile_name": f"irq_{zone}N_E{int(x0/1000):04d}_N{int(y0/1000):04d}",
                      "utm_zone": zone, "epsg_utm": epsg, "geometry_utm_wkt": utm_box.wkt,
                      "farmland_km2": round(float(fkm2), 1),
                      "area_km2": round(area, 1), "est_credits": round(est_credits(area), 1),
                      "geometry": geom})

grid = gpd.GeoDataFrame(tiles, crs=4326)
print(f"\nTiles kept: {len(grid)}  (squares of {cfg['tile_km']} km with ≥ {MIN_FARM_KM2} km² farmland, shrunk to it)")

# ---- Winter season per tile from WorldCereal's crop calendar ----------------
from worldcereal.seasons import enrich_production_grid_from_crop_calendars
g2 = grid[["tile_name", "geometry"]].copy()
g2["epsg"] = 4326
enr = enrich_production_grid_from_crop_calendars(g2, year=YEAR, seasons=("tc-s1",), on_heterogeneity="warn")
enr = enr.set_index("tile_name")
grid["start_date"] = grid["tile_name"].map(enr["start_date"]).astype(str)
grid["end_date"] = grid["tile_name"].map(enr["end_date"]).astype(str)
sw = grid["tile_name"].map(enr["season_windows"]).map(lambda s: json.loads(s) if isinstance(s, str) else s)
grid["season_start_tc-s1"] = sw.map(lambda d: d["tc-s1"][0])
grid["season_end_tc-s1"] = sw.map(lambda d: d["tc-s1"][1])
grid["calendar_heterogeneous"] = grid["tile_name"].map(enr.get("seasonality_heterogeneous", pd.Series(dtype=bool))).fillna(False).astype(bool)
today = date.today().isoformat()
future = grid[grid["end_date"] >= today]
if len(future):
    raise SystemExit(f"{len(future)} tile(s) have a season ending on/after today ({today}) — choose an earlier season_year.")

grid = grid.sort_values("farmland_km2", ascending=False).reset_index(drop=True)
grid.to_file(os.path.join(PLAN, "grid_all.geojson"), driver="GeoJSON")

seasons = grid.groupby(["season_start_tc-s1", "season_end_tc-s1"]).size().to_dict()
summary = {
    "tiles": int(len(grid)),
    "farmland_km2": round(float(grid["farmland_km2"].sum())),
    "processed_area_km2": round(float(grid["area_km2"].sum())),
    "est_credits_total": round(float(grid["est_credits"].sum())),
    "season_windows": {f"{a} → {b}": int(n) for (a, b), n in seasons.items()},
    "heterogeneous_calendar_tiles": int(grid["calendar_heterogeneous"].sum()),
    "settings": {k: cfg[k] for k in ("tile_km", "farmland_conf", "min_farmland_km2", "season_year")},
}
json.dump(summary, open(os.path.join(PLAN, "plan.json"), "w"), indent=1)
print(json.dumps(summary, indent=1))

# ---- Plan map ----------------------------------------------------------------
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 8), dpi=120)
    iraq.boundary.plot(ax=ax, color="#14261B", linewidth=0.8)
    ax.scatter(farm_lon, farm_lat, s=0.05, c="#7fa86a", linewidths=0)
    grid.boundary.plot(ax=ax, color="#d9a441", linewidth=0.8)
    for _, r in grid.head(12).iterrows():
        c = r.geometry.centroid
        ax.annotate(f"{r.est_credits:.0f}", (c.x, c.y), fontsize=5, ha="center", color="#6b4f13")
    ax.set_title(f"{len(grid)} tiles · ~{summary['processed_area_km2']:,} km² processed · ~{summary['est_credits_total']:,} credits")
    ax.set_aspect(1 / math.cos(math.radians((S + N) / 2)))
    ax.set_axis_off()
    fig.savefig(os.path.join(PLAN, "plan_map.png"), bbox_inches="tight")
    print("Map:", os.path.join(PLAN, "plan_map.png"))
except Exception as e:
    print("(map skipped:", e, ")")

In [ ]:
%%writefile /content/batch_tool.py
"""Batches, status, diagnosis and compaction for the Iraq WorldCereal run.

python batch_tool.py next     <out> <credit_budget>   -> prints the batch folder to run (new or resumed)
python batch_tool.py status   <out>
python batch_tool.py diagnose <out>
python batch_tool.py compact  <out> [--delete-raw]

Layout under <out>:
  plan/grid_all.geojson         every tile, with its season and credit estimate
  batch_01/grid.geojson         the tiles of one batch (fits the credit budget)
  batch_01/job_tracking.csv     WorldCereal's own job database (resume state)
  batch_01/<tile>/*.tif         raw WorldCereal output (29 bands)
  compact/<tile>.tif            4-band compact copy kept for the app
"""
import csv
import glob
import json
import os
import sys

import geopandas as gpd
import pandas as pd

DONE = "finished"
TERMINAL = {"finished", "error", "start_failed", "canceled", "skipped"}
MAX_ATTEMPTS = 2           # a tile that fails in two separate batches is set aside, not retried forever
GRID_COLS = ["tile_name", "epsg_utm", "geometry_utm_wkt", "start_date", "end_date",
             "season_start_tc-s1", "season_end_tc-s1", "geometry"]


def batches(out):
    return sorted(d for d in glob.glob(os.path.join(out, "batch_*")) if os.path.isdir(d))


def tracking(batch_dir):
    p = os.path.join(batch_dir, "job_tracking.csv")
    return list(csv.DictReader(open(p, newline=""))) if os.path.exists(p) else []


def tile_history(out):
    """tile -> list of statuses across batches (oldest first)."""
    hist = {}
    for b in batches(out):
        for r in tracking(b):
            hist.setdefault(r["tile_name"], []).append(r.get("status") or "not_started")
    return hist


def cmd_next(out, budget):
    plan = gpd.read_file(os.path.join(out, "plan", "grid_all.geojson"))
    bs = batches(out)
    if bs:
        last = bs[-1]
        rows = tracking(last)
        if not rows or any((r.get("status") or "not_started") not in TERMINAL for r in rows):
            print(f"RESUME {last}")
            return last
        failed = [r["tile_name"] for r in rows if r.get("status") != DONE]
        if failed:
            print(f"(previous batch {os.path.basename(last)} ended with {len(failed)} failed tile(s); they're queued for one retry)")
    hist = tile_history(out)
    done = {t for t, s in hist.items() if DONE in s}
    gave_up = {t for t, s in hist.items() if DONE not in s and sum(x != DONE for x in s) >= MAX_ATTEMPTS}
    todo = plan[~plan["tile_name"].isin(done | gave_up)]          # plan is sorted most-farmland-first
    if todo.empty:
        print(f"ALL_DONE {len(done)} tiles finished" + (f", {len(gave_up)} set aside after repeated failures" if gave_up else ""))
        return None
    pick, total = [], 0.0
    for _, r in todo.iterrows():
        if pick and total + r["est_credits"] > budget:
            break
        pick.append(r)
        total += r["est_credits"]
    batch = gpd.GeoDataFrame(pick, crs=plan.crs)[GRID_COLS]
    bdir = os.path.join(out, f"batch_{len(bs) + 1:02d}")
    os.makedirs(bdir, exist_ok=True)
    batch.to_file(os.path.join(bdir, "grid.geojson"), driver="GeoJSON")
    print(f"NEW {bdir}  — {len(batch)} tiles, ~{total:,.0f} credits (budget {budget:,.0f}); "
          f"{len(todo) - len(batch)} tiles left after this batch")
    return bdir


def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0


def cmd_status(out):
    plan = gpd.read_file(os.path.join(out, "plan", "grid_all.geojson"))
    hist = tile_history(out)
    spent = 0.0
    for b in batches(out):
        rows = tracking(b)
        cnt = pd.Series([r.get("status") or "not_started" for r in rows]).value_counts().to_dict()
        c = sum(_f(r.get("costs")) for r in rows)
        spent += c
        print(f"{os.path.basename(b)}: {len(rows)} tiles  {cnt}  credits {c:,.0f}")
    done = {t for t, s in hist.items() if DONE in s}
    left = plan[~plan["tile_name"].isin(done)]
    compact = len(glob.glob(os.path.join(out, "compact", "*.tif")))
    print(f"\nTiles finished: {len(done)} / {len(plan)}   compacted: {compact}")
    print(f"Credits used so far: {spent:,.0f}   estimated still needed: ~{left['est_credits'].sum():,.0f}")
    print(f"Farmland covered: {plan[plan['tile_name'].isin(done)]['farmland_km2'].sum():,.0f} / {plan['farmland_km2'].sum():,.0f} km²")


def cmd_diagnose(out):
    found = False
    for b in batches(out):
        for r in tracking(b):
            if r.get("status") in ("error", "start_failed"):
                found = True
                jid = r.get("id")
                print(f"\n=== {os.path.basename(b)} / {r['tile_name']} | job {jid} | credits {r.get('costs') or '?'}")
                saved = glob.glob(os.path.join(b, f"job_{jid}", "*errors.json"))
                logs = []
                if saved:
                    logs = json.load(open(saved[0]))
                elif jid:
                    try:
                        from openeo_gfmap.backend import cdse_connection
                        logs = [dict(l) for l in cdse_connection().job(jid).logs(level="error")]
                    except Exception as e:
                        print("   (couldn't fetch logs:", e, ")")
                for l in logs[-10:]:
                    print("  -", str(l.get("message", l))[:1200])
                if not logs:
                    print("   (no error messages recorded — see https://openeo.dataspace.copernicus.eu/)")
    if not found:
        print("No failed jobs.")


def cmd_compact(out, delete_raw):
    """Keep what the app needs: class, its confidence, P(wheat), P(barley). ~1/7 of the raw size."""
    import numpy as np
    import rasterio
    cdir = os.path.join(out, "compact")
    os.makedirs(cdir, exist_ok=True)
    n_new = freed = 0
    legend = None
    for b in batches(out):
        finished = {r["tile_name"] for r in tracking(b) if r.get("status") == DONE}
        for tile in sorted(finished):
            dst = os.path.join(cdir, f"{tile}.tif")
            raws = glob.glob(os.path.join(b, tile, "*.tif"))
            if os.path.exists(dst):
                if delete_raw:
                    for f in raws:
                        freed += os.path.getsize(f); os.remove(f)
                continue
            if not raws:
                print(f"  {tile}: no raw GeoTIFF found (not downloaded?)")
                continue
            with rasterio.open(raws[0]) as ds:
                names = list(ds.descriptions)
                season = next(n.split(":")[1] for n in names if n.startswith("croptype_classification:"))
                classes = [n.split(":")[2] for n in names if n.startswith(f"croptype_probability:{season}:")]
                legend = legend or classes
                want = [f"croptype_classification:{season}", f"croptype_probability:{season}",
                        f"croptype_probability:{season}:wheat", f"croptype_probability:{season}:barley"]
                arrs = [ds.read(names.index(w) + 1) for w in want]
                prof = ds.profile.copy()
            prof.update(driver="COG", count=4, dtype="uint8", compress="DEFLATE", predictor=2,
                        nodata=255, BIGTIFF="IF_SAFER")
            for k in ("blockxsize", "blockysize", "tiled", "interleave", "photometric"):
                prof.pop(k, None)
            with rasterio.open(dst, "w", **prof) as o:
                for i, (a, nm) in enumerate(zip(arrs, ["class", "class_confidence", "p_wheat", "p_barley"]), 1):
                    o.write(a.astype("uint8"), i)
                    o.set_band_description(i, nm)
                o.update_tags(season=season, classes=json.dumps(classes), not_cropland_value="254",
                              source="ESA WorldCereal default model, run via worldcereal-classification")
            n_new += 1
            print(f"  {tile}: {os.path.getsize(raws[0])/1e6:.0f} MB → {os.path.getsize(dst)/1e6:.1f} MB")
            if delete_raw:
                for f in raws:
                    freed += os.path.getsize(f); os.remove(f)
    if legend:
        json.dump({"bands": ["class", "class_confidence", "p_wheat", "p_barley"], "class_codes": dict(enumerate(legend)),
                   "not_cropland": 254, "nodata": 255, "probability_scale": "0-100"},
                  open(os.path.join(cdir, "legend.json"), "w"), indent=1)
    print(f"Compacted {n_new} new tile(s)" + (f"; freed {freed/1e9:.2f} GB of raw files" if delete_raw else ""))


if __name__ == "__main__":
    cmd, out = sys.argv[1], sys.argv[2]
    if cmd == "next":
        cmd_next(out, float(sys.argv[3]))
    elif cmd == "status":
        cmd_status(out)
    elif cmd == "diagnose":
        cmd_diagnose(out)
    elif cmd == "compact":
        cmd_compact(out, "--delete-raw" in sys.argv)
    else:
        sys.exit(f"unknown command {cmd}")

In [ ]:
REPLAN = False
if REPLAN or not os.path.exists(os.path.join(OUT, "plan", "grid_all.geojson")):
    sh(f"/content/wcenv/bin/python -u /content/plan_tiles.py {shlex.quote(CFG)}")
else:
    print("Using the existing plan:"); print(open(os.path.join(OUT, "plan", "plan.json")).read())
from IPython.display import Image, display
display(Image(os.path.join(OUT, "plan", "plan_map.png"), width=650))

## 5 · Run the next batch (hours — safe to resume)
Picks the next tiles that fit `CREDIT_BUDGET` (most farmland first), or **resumes** the unfinished batch. When the batch ends it shrinks the results and prints the status. Run this again for the next batch.

If you stop it on purpose, the cloud jobs keep running — cancel them at [openeo.dataspace.copernicus.eu](https://openeo.dataspace.copernicus.eu/) if you want them stopped.

In [ ]:
msg = sh(f"/content/wcenv/bin/python -u /content/batch_tool.py next {shlex.quote(OUT)} {CREDIT_BUDGET}")
line = [l for l in msg.splitlines() if l.startswith(("NEW ", "RESUME ", "ALL_DONE"))][-1]
if line.startswith("ALL_DONE"):
    print("\n✅ Every planned tile is done — nothing to submit.")
else:
    BATCH = line.split()[1]
    args = [
        "--task", "classification",
        "--grid_path", os.path.join(BATCH, "grid.geojson"), "--grid_size", "none",
        "--product", "croptype",
        "--enable-cropland-head", "--enable-croptype-head", "--mask-cropland",
        "--enable-cropland-postprocess", "--cropland-postprocess-method", "majority_vote", "--cropland-postprocess-kernel-size", "3",
        "--enable-croptype-postprocess", "--croptype-postprocess-method", "majority_vote", "--croptype-postprocess-kernel-size", "5",
        "--class-probabilities", "--merge-classification-products",
        "--parallel_jobs", str(PARALLEL_JOBS),
        "--output_folder", BATCH,
    ]
    sh("cd /content/worldcereal-classification && /content/wcenv/bin/python -u scripts/inference/run_worldcereal_task_openeo.py "
       + " ".join(shlex.quote(a) for a in args))
    sh(f"/content/wcenv/bin/python -u /content/batch_tool.py compact {shlex.quote(OUT)}" + (" --delete-raw" if DELETE_RAW_AFTER_COMPACT else ""))
    sh(f"/content/wcenv/bin/python -u /content/batch_tool.py status {shlex.quote(OUT)}")

## 6 · Status (any time)
Tiles done, credits used and still needed, farmland covered.

In [ ]:
sh(f"/content/wcenv/bin/python -u /content/batch_tool.py status {shlex.quote(OUT)}")

## 7 · If tiles failed: show why
A failed tile is retried once in the next batch automatically, then set aside. This prints the error messages — send them to me.

In [ ]:
sh(f"/content/wcenv/bin/python -u /content/batch_tool.py diagnose {shlex.quote(OUT)}")

## 8 · Package for checking
Zips the plan, the job tracking files and the compact tiles finished so far. Early on it's small — send it to me after the first batch so I can check a few tiles across the country before you spend the rest of the credits.

In [ ]:
import glob, zipfile
zpath = f"/content/{RUN_NAME}_progress.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(OUT, "plan", "*")) + glob.glob(os.path.join(OUT, "batch_*", "job_tracking.csv")) \
             + glob.glob(os.path.join(OUT, "batch_*", "grid.geojson")) + glob.glob(os.path.join(OUT, "compact", "*")):
        z.write(f, os.path.relpath(f, OUT))
print(f"{zpath}  ({os.path.getsize(zpath)/1e6:.1f} MB)")
if os.path.getsize(zpath) < 500e6:
    from google.colab import files
    files.download(zpath)
else:
    print("Large file — share it from Drive instead:", shutil.copy(zpath, OUT))